## RestaurantAssistAI: LLM-Powered Restaurant Recommendation Chatbot

### **Project Overview**

RestaurantAssistAI is a conversational AI application that helps users discover restaurants based on natural language preferences such as cuisine, location, budget, ambience, parking availability, dietary preferences, and review-based indicators.</br>
The system uses OpenAI APIs and LangChain to understand user requirements, filter restaurant options from a structured dataset, and generate personalized recommendations with reasoning.

#### **Business Problem**:
Users often struggle to identify restaurants that match their specific preferences across cuisine, budget, location, ambience, and occasion.
</br>
The objective of this project is to build an LLM-powered chatbot that can interact with users conversationally, understand their requirements, retrieve relevant restaurant options, and recommend the best-fit restaurant with a persuasive explanation.

#### **Data Sourcing**:
Publicly available zomato database of 'Bangalore' is used as a datasource for this application, that includes the comprehensive list of all the restaurants in Bangalore with their address details, cost, ratings and user reviews.

#### **Scope**:
The application was tested end-to-end on a representative subset of 1,000 restaurant records due to API cost and dataset size constraints.

#### **Project Objectives**:
1. Clean and prepare restaurant review data.
2. Extract useful restaurant attributes from user reviews using OpenAI API.
3. Build a conversational flow to collect user preferences.
4. Use LangChain and OpenAI to filter restaurants based on natural language requirements.
5. Handle unavailable user-entered locations using geopy-based nearby-location search.
6. Recommend the best restaurant from shortlisted candidates.
7. Generate an explanation supporting the recommendation.

#### **Design**:
Creating this chatbot application is mainly classified into 4 stages:
</br>
</br>
*Stage1*: Data sourcing and preparation - Data collected from public source has to be cleaned and formatted  before use. Also, during this stage required data pointers like ambience, parking availability and suggested food items are extracted from user reviews_list column using OpenAI API to understand the user reviews and extract the values for these features.
</br>
</br>
*Stage2*: User requirement and intent confirmation - In this stage, we use OpenAI API to understand the user requirement through a series of conversations and confirming that user preference is obtained for all the features that we are expecting.
</br>
</br>
*Stage3*: Data extraction - Here, we use LangChain and OpenAI to filter the list of Top 5 restaurants that caters to the user query. (LangChain+OpenAI is used to filter the data from dataframe). If no restaurants are found for the input location, this layer has the intelligence to extract the matching restaurant list from the nearby location(within 3 km radius from the user input location). This is achieved using the geocoding library 'geopy'.
</br>
</br>
*Stage4*: Restaurant Recommendation - Among the top 5 restaurants returned from Data extraction layer, this layer will make recommendation on one of the restaurant (using OpenAI API) with the proper reasoning. This makes the user feel-good and also influencing the user decision in chosing the restaurant.
</br>
</br>
Note that Moderation API is wrapped around both Input and Output data so as to make this application secure.

#### **Output**:
1. List of top 5 restaurants that satisfies the user requirements
2. Recommendation of one restaurant with proper reasoning that can help user in making quick decision

### **Building Restaurant recommender AI assistant**

### Installing required dependencies and importing the libraries

Install the required dependencies

In [38]:
!pip install langchain

In [39]:
!pip install langchain_experimental

In [40]:
!pip install langchain_openai

In [41]:
!pip install geopy

Import the required libraries

In [42]:
import pandas as pd
import numpy as np
import os, json
import openai
import re

import warnings
warnings.filterwarnings('ignore')

from tenacity import retry, wait_random_exponential, stop_after_attempt
from langchain.agents.agent_types import AgentType
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain_openai import ChatOpenAI

from geopy.geocoders import Nominatim
from geopy.geocoders import Photon
from geopy.distance import distance

In [43]:
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

import openai
openai.api_key = OPENAI_API_KEY

### 1. Data Sourcing

In [ ]:
zomato_data=pd.read_csv("../data/zomato.csv", engine='python', encoding='utf-8', on_bad_lines='skip')

In [ ]:
zomato_df=zomato_data.copy()
zomato_df.head(2)

,url,address,name,online_order,book_table,rate,votes,phone,location,rest_type,dish_liked,cuisines,approx_cost(for two people),reviews_list,menu_item,listed_in(type),listed_in(city)
0,https://www.zomato.com/bangalore/jalsa-banasha...,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,Yes,Yes,4.1/5,775,080 42297555\r\n+91 9743772233,Banashankari,Casual Dining,"Pasta, Lunch Buffet, Masala Papad, Paneer Laja...","North Indian, Mughlai, Chinese",800,"[('Rated 4.0', 'RATED\n A beautiful place to ...",[],Buffet,Banashankari
1,https://www.zomato.com/bangalore/spice-elephan...,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,Yes,No,4.1/5,787,080 41714161,Banashankari,Casual Dining,"Momos, Lunch Buffet, Chocolate Nirvana, Thai G...","Chinese, North Indian, Thai",800,"[('Rated 4.0', 'RATED\n Had been here for din...",[],Buffet,Banashankari


In [ ]:
zomato_df.shape

(51155, 17)

In [ ]:
zomato_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51155 entries, 0 to 51154
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   url                          51155 non-null  object
 1   address                      51155 non-null  object
 2   name                         51155 non-null  object
 3   online_order                 51155 non-null  object
 4   book_table                   51155 non-null  object
 5   rate                         43380 non-null  object
 6   votes                        51155 non-null  int64 
 7   phone                        49952 non-null  object
 8   location                     51134 non-null  object
 9   rest_type                    50931 non-null  object
 10  dish_liked                   23081 non-null  object
 11  cuisines                     51110 non-null  object
 12  approx_cost(for two people)  50811 non-null  object
 13  reviews_list                 51

In [ ]:
# Check the Null Values
print(zomato_df.isnull().sum())

url                                0
address                            0
name                               0
online_order                       0
book_table                         0
rate                            7775
votes                              0
phone                           1203
location                          21
rest_type                        224
dish_liked                     28074
cuisines                          45
approx_cost(for two people)      344
reviews_list                       0
menu_item                          0
listed_in(type)                    0
listed_in(city)                    0
dtype: int64


### 2. Data Cleaning/Data Preparation

Dropping the column "dish_liked" and "url". Popular dishes are extracted from reviews_list and URL is referrfing to old URL
which is not valid now. Hence dropping these two coloumns. Same case with phone number as well

##### 2.1 Drop unnecessary columns

In [ ]:
zomato_df=zomato_df.drop(['phone','dish_liked','url'],axis=1)



```
# This is formatted as code
```

##### 2.2 Handling NULL values

In [ ]:
#Remove the NaN values from the dataset
zomato_df.dropna(how='any',inplace=True)

##### 2.3 Renaming columns for better interpretability

In [ ]:
#Changing the column names
zomato_df = zomato_df.rename(columns={'approx_cost(for two people)':'cost','listed_in(type)':'type', 'listed_in(city)':'city'})

##### 2.4 Analyzing 'rate' column

In [ ]:
#Analyzing rate column
zomato_df['rate'].value_counts()

,count
rate,
NEW,2203
3.9/5,2086
3.7/5,2003
3.8/5,1994
3.9 /5,1852
...,...
2.0 /5,7
2.1 /5,7
2.0/5,4


In [ ]:
#Removing '/5' from Rates
zomato_df = zomato_df.loc[zomato_df.rate !='NEW']
zomato_df = zomato_df.loc[zomato_df.rate !='-'].reset_index(drop=True)
remove_slash = lambda x: x.replace('/5', '')
zomato_df.rate = zomato_df.rate.apply(remove_slash).str.strip().astype('float')

##### 2.5 Analyzing 'cost' column

In [ ]:
#Changing the cost to string
zomato_df['cost'] = zomato_df['cost'].astype(str)
zomato_df['cost'] = zomato_df['cost'].apply(lambda x: x.replace(',','.'))
zomato_df['cost'] = zomato_df['cost'].astype(float)

In [ ]:
#After Cleaning
zomato_df.shape

(40708, 14)

##### 2.6 Restaurant name - Cleaning

In [ ]:
restaurants = list(zomato_df['name'].unique())
restaurants[:20]

['Jalsa',
 'Spice Elephant',
 'San Churro Cafe',
 'Addhuri Udupi Bhojana',
 'Grand Village',
 'Timepass Dinner',
 'Rosewood International Hotel - Bar & Restaurant',
 'Onesta',
 'Penthouse Cafe',
 'Smacznego',
 'CafÃ\x83Â\x83Ã\x82Â\x83Ã\x83Â\x82Ã\x82Â\x83Ã\x83Â\x83Ã\x82Â\x82Ã\x83Â\x82Ã\x82Â© Down The Alley',
 'Cafe Shuffle',
 'The Coffee Shack',
 'Caf-Eleven',
 'Cafe Vivacity',
 'Catch-up-ino',
 "Kirthi's Biryani",
 'T3H Cafe',
 '360 Atoms Restaurant And Cafe',
 'The Vintage Cafe']

Notice the presence of some Non ASCII characters, which need to be cleaned

In [ ]:
# Define the regular expression pattern
pattern = r"[^a-zA-Z0-9\s\-\'\"]"

In [ ]:
zomato_df['name'] = zomato_df['name'].str.replace(pattern, '', regex=True)

In [ ]:
restaurants = list(zomato_df['name'].unique())
restaurants[:20]

['Jalsa',
 'Spice Elephant',
 'San Churro Cafe',
 'Addhuri Udupi Bhojana',
 'Grand Village',
 'Timepass Dinner',
 'Rosewood International Hotel - Bar  Restaurant',
 'Onesta',
 'Penthouse Cafe',
 'Smacznego',
 'Caf Down The Alley',
 'Cafe Shuffle',
 'The Coffee Shack',
 'Caf-Eleven',
 'Cafe Vivacity',
 'Catch-up-ino',
 "Kirthi's Biryani",
 'T3H Cafe',
 '360 Atoms Restaurant And Cafe',
 'The Vintage Cafe']

##### 2.7 review list - Cleaning

In [ ]:
pd.set_option('display.max_colwidth', None)

In [ ]:
zomato_df['reviews_list'].head(1)

,reviews_list
0,"[('Rated 4.0', 'RATED\n A beautiful place to dine in.The interiors take you back to the Mughal era. The lightings are just perfect.We went there on the occasion of Christmas and so they had only limited items available. But the taste and service was not compromised at all.The only complaint is that the breads could have been better.Would surely like to come here again.'), ('Rated 4.0', 'RATED\n I was here for dinner with my family on a weekday. The restaurant was completely empty. Ambience is good with some good old hindi music. Seating arrangement are good too. We ordered masala papad, panner and baby corn starters, lemon and corrionder soup, butter roti, olive and chilli paratha. Food was fresh and good, service is good too. Good for family hangout.\nCheers'), ('Rated 2.0', 'RATED\n Its a restaurant near to Banashankari BDA. Me along with few of my office friends visited to have buffet but unfortunately they only provide veg buffet. On inquiring they said this place is mostly visited by vegetarians. Anyways we ordered ala carte items which took ages to come. Food was ok ok. Definitely not visiting anymore.'), ('Rated 4.0', 'RATED\n We went here on a weekend and one of us had the buffet while two of us took Ala Carte. Firstly the ambience and service of this place is great! The buffet had a lot of items and the good was good. We had a Pumpkin Halwa intm the dessert which was amazing. Must try! The kulchas are great here. Cheers!'), ('Rated 5.0', 'RATED\n The best thing about the place is itÃ\x83Ã\x83Ã\x82Ã\x82Ã\x83Ã\x82Ã\x82Ã\x92s ambiance. Second best thing was yummy ? food. We try buffet and buffet food was not disappointed us.\nTest ?. ?? ?? ?? ?? ??\nQuality ?. ??????????.\nService: Staff was very professional and friendly.\n\nOverall experience was excellent.\n\nsubirmajumder85.wixsite.com'), ('Rated 5.0', 'RATED\n Great food and pleasant ambience. Expensive but Coll place to chill and relax......\n\nService is really very very good and friendly staff...\n\nFood : 5/5\nService : 5/5\nAmbience :5/5\nOverall :5/5'), ('Rated 4.0', 'RATED\n Good ambience with tasty food.\nCheese chilli paratha with Bhutta palak methi curry is a good combo.\nLemon Chicken in the starters is a must try item.\nEgg fried rice was also quite tasty.\nIn the mocktails, recommend ""Alice in Junoon"". Do not miss it.'), ('Rated 4.0', 'RATED\n You canÃ\x83Ã\x83Ã\x82Ã\x82Ã\x83Ã\x82Ã\x82Ã\x92t go wrong with Jalsa. Never been a fan of their buffet and thus always order alacarteÃ\x83Ã\x83Ã\x82Ã\x82Ã\x83Ã\x82Ã\x82Ã\x92. Service at times can be on the slower side but food is worth the wait.'), ('Rated 5.0', 'RATED\n Overdelighted by the service and food provided at this place. A royal and ethnic atmosphere builds a strong essence of being in India and also the quality and taste of food is truly authentic. I would totally recommend to visit this place once.'), ('Rated 4.0', 'RATED\n The place is nice and comfortable. Food wise all jalea outlets maintain a good standard. The soya chaap was a standout dish. Clearly one of trademark dish as per me and a must try.\n\nThe only concern is the parking. It very congested and limited to just 5cars. The basement parking is very steep and makes it cumbersome'), ('Rated 4.0', 'RATED\n The place is nice and comfortable. Food wise all jalea outlets maintain a good standard. The soya chaap was a standout dish. Clearly one of trademark dish as per me and a must try.\n\nThe only concern is the parking. It very congested and limited to just 5cars. The basement parking is very steep and makes it cumbersome'), ('Rated 4.0', 'RATED\n The place is nice and comfortable. Food wise all jalea outlets maintain a good standard. The soya chaap was a standout dish. Clearly one of trademark dish as per me and a must try.\n\nThe only concern is the parking. It very congested and limited to just 5cars. The basement parking is very steep and makes it cumbersome')]"


As we could see there are some 'Non-ASCII' charecters in the reviews column which needs to be cleaned

In [ ]:
series = zomato_df['reviews_list']
zomato_df["reviews_list"] =  [s.encode('ascii', 'ignore').strip()
               for s in series.str.decode('unicode_escape')]

In [ ]:
zomato_df['reviews_list'].head(1)

,reviews_list
0,"b'[(\'Rated 4.0\', \'RATED\n A beautiful place to dine in.The interiors take you back to the Mughal era. The lightings are just perfect.We went there on the occasion of Christmas and so they had only limited items available. But the taste and service was not compromised at all.The only complaint is that the breads could have been better.Would surely like to come here again.\'), (\'Rated 4.0\', \'RATED\n I was here for dinner with my family on a weekday. The restaurant was completely empty. Ambience is good with some good old hindi music. Seating arrangement are good too. We ordered masala papad, panner and baby corn starters, lemon and corrionder soup, butter roti, olive and chilli paratha. Food was fresh and good, service is good too. Good for family hangout.\nCheers\'), (\'Rated 2.0\', \'RATED\n Its a restaurant near to Banashankari BDA. Me along with few of my office friends visited to have buffet but unfortunately they only provide veg buffet. On inquiring they said this place is mostly visited by vegetarians. Anyways we ordered ala carte items which took ages to come. Food was ok ok. Definitely not visiting anymore.\'), (\'Rated 4.0\', \'RATED\n We went here on a weekend and one of us had the buffet while two of us took Ala Carte. Firstly the ambience and service of this place is great! The buffet had a lot of items and the good was good. We had a Pumpkin Halwa intm the dessert which was amazing. Must try! The kulchas are great here. Cheers!\'), (\'Rated 5.0\', \'RATED\n The best thing about the place is its ambiance. Second best thing was yummy ? food. We try buffet and buffet food was not disappointed us.\nTest ?. ?? ?? ?? ?? ??\nQuality ?. ??????????.\nService: Staff was very professional and friendly.\n\nOverall experience was excellent.\n\nsubirmajumder85.wixsite.com\'), (\'Rated 5.0\', \'RATED\n Great food and pleasant ambience. Expensive but Coll place to chill and relax......\n\nService is really very very good and friendly staff...\n\nFood : 5/5\nService : 5/5\nAmbience :5/5\nOverall :5/5\'), (\'Rated 4.0\', \'RATED\n Good ambience with tasty food.\nCheese chilli paratha with Bhutta palak methi curry is a good combo.\nLemon Chicken in the starters is a must try item.\nEgg fried rice was also quite tasty.\nIn the mocktails, recommend ""Alice in Junoon"". Do not miss it.\'), (\'Rated 4.0\', \'RATED\n You cant go wrong with Jalsa. Never been a fan of their buffet and thus always order alacarte. Service at times can be on the slower side but food is worth the wait.\'), (\'Rated 5.0\', \'RATED\n Overdelighted by the service and food provided at this place. A royal and ethnic atmosphere builds a strong essence of being in India and also the quality and taste of food is truly authentic. I would totally recommend to visit this place once.\'), (\'Rated 4.0\', \'RATED\n The place is nice and comfortable. Food wise all jalea outlets maintain a good standard. The soya chaap was a standout dish. Clearly one of trademark dish as per me and a must try.\n\nThe only concern is the parking. It very congested and limited to just 5cars. The basement parking is very steep and makes it cumbersome\'), (\'Rated 4.0\', \'RATED\n The place is nice and comfortable. Food wise all jalea outlets maintain a good standard. The soya chaap was a standout dish. Clearly one of trademark dish as per me and a must try.\n\nThe only concern is the parking. It very congested and limited to just 5cars. The basement parking is very steep and makes it cumbersome\'), (\'Rated 4.0\', \'RATED\n The place is nice and comfortable. Food wise all jalea outlets maintain a good standard. The soya chaap was a standout dish. Clearly one of trademark dish as per me and a must try.\n\nThe only concern is the parking. It very congested and limited to just 5cars. The basement parking is very steep and makes it cumbersome\')]'"


Remove \n for better readability

In [ ]:
zomato_df['reviews_list'] = zomato_df['reviews_list'].str.decode('utf-8').replace('\n', '')
zomato_df['reviews_list'] = zomato_df['reviews_list'].apply(lambda x: x.replace('\n', ''))

In [ ]:
zomato_df['reviews_list'].head(1)

,reviews_list
0,"[('Rated 4.0', 'RATED A beautiful place to dine in.The interiors take you back to the Mughal era. The lightings are just perfect.We went there on the occasion of Christmas and so they had only limited items available. But the taste and service was not compromised at all.The only complaint is that the breads could have been better.Would surely like to come here again.'), ('Rated 4.0', 'RATED I was here for dinner with my family on a weekday. The restaurant was completely empty. Ambience is good with some good old hindi music. Seating arrangement are good too. We ordered masala papad, panner and baby corn starters, lemon and corrionder soup, butter roti, olive and chilli paratha. Food was fresh and good, service is good too. Good for family hangout.Cheers'), ('Rated 2.0', 'RATED Its a restaurant near to Banashankari BDA. Me along with few of my office friends visited to have buffet but unfortunately they only provide veg buffet. On inquiring they said this place is mostly visited by vegetarians. Anyways we ordered ala carte items which took ages to come. Food was ok ok. Definitely not visiting anymore.'), ('Rated 4.0', 'RATED We went here on a weekend and one of us had the buffet while two of us took Ala Carte. Firstly the ambience and service of this place is great! The buffet had a lot of items and the good was good. We had a Pumpkin Halwa intm the dessert which was amazing. Must try! The kulchas are great here. Cheers!'), ('Rated 5.0', 'RATED The best thing about the place is its ambiance. Second best thing was yummy ? food. We try buffet and buffet food was not disappointed us.Test ?. ?? ?? ?? ?? ??Quality ?. ??????????.Service: Staff was very professional and friendly.Overall experience was excellent.subirmajumder85.wixsite.com'), ('Rated 5.0', 'RATED Great food and pleasant ambience. Expensive but Coll place to chill and relax......Service is really very very good and friendly staff...Food : 5/5Service : 5/5Ambience :5/5Overall :5/5'), ('Rated 4.0', 'RATED Good ambience with tasty food.Cheese chilli paratha with Bhutta palak methi curry is a good combo.Lemon Chicken in the starters is a must try item.Egg fried rice was also quite tasty.In the mocktails, recommend ""Alice in Junoon"". Do not miss it.'), ('Rated 4.0', 'RATED You cant go wrong with Jalsa. Never been a fan of their buffet and thus always order alacarte. Service at times can be on the slower side but food is worth the wait.'), ('Rated 5.0', 'RATED Overdelighted by the service and food provided at this place. A royal and ethnic atmosphere builds a strong essence of being in India and also the quality and taste of food is truly authentic. I would totally recommend to visit this place once.'), ('Rated 4.0', 'RATED The place is nice and comfortable. Food wise all jalea outlets maintain a good standard. The soya chaap was a standout dish. Clearly one of trademark dish as per me and a must try.The only concern is the parking. It very congested and limited to just 5cars. The basement parking is very steep and makes it cumbersome'), ('Rated 4.0', 'RATED The place is nice and comfortable. Food wise all jalea outlets maintain a good standard. The soya chaap was a standout dish. Clearly one of trademark dish as per me and a must try.The only concern is the parking. It very congested and limited to just 5cars. The basement parking is very steep and makes it cumbersome'), ('Rated 4.0', 'RATED The place is nice and comfortable. Food wise all jalea outlets maintain a good standard. The soya chaap was a standout dish. Clearly one of trademark dish as per me and a must try.The only concern is the parking. It very congested and limited to just 5cars. The basement parking is very steep and makes it cumbersome')]"


Final Data set after all data cleaning

In [ ]:
pd.reset_option('^display.', silent=True)

In [ ]:
zomato_df.head(2)

,address,name,online_order,book_table,rate,votes,location,rest_type,cuisines,cost,reviews_list,menu_item,type,city
0,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,Yes,Yes,4.1,775,Banashankari,Casual Dining,"North Indian, Mughlai, Chinese",800.0,"[('Rated 4.0', 'RATED A beautiful place to di...",[],Buffet,Banashankari
1,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,Yes,No,4.1,787,Banashankari,Casual Dining,"Chinese, North Indian, Thai",800.0,"[('Rated 4.0', 'RATED Had been here for dinne...",[],Buffet,Banashankari


Notice the 'cost' and 'rate' columns above which has been formatted

### 3. Data extraction

##### 3.1 Using OpenAI API to extract the list of user reviews from reviews_list column

From the above dataset, we focus mainly on reviews_list column and capture some important info from the same. Note that this is done at once and the extracted info is added as new column to the dataset, so that only extracted, refined, required data will be passed as input while filtering the data for user queries from OpenAI API.

In [23]:
@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(6))
def get_chat_completions(input, json_format = False):
    MODEL = 'gpt-4o-mini'

    system_message_json_output = """<<. Return output in JSON format to the key output.>>"""

    # If the output is required to be in JSON format
    if json_format == True:
        # Append the input prompt to include JSON response as specified by OpenAI
        input[0]['content'] += system_message_json_output

        # JSON return type specified
        chat_completion_json = openai.chat.completions.create(
            model = MODEL,
            messages = input,
            response_format = { "type": "json_object"},
            seed = 1234)

        output = json.loads(chat_completion_json.choices[0].message.content)

    # No JSON return type specified
    else:
        chat_completion = openai.chat.completions.create(
            model = MODEL,
            messages = input,
            seed = 2345)

        output = chat_completion.choices[0].message.content

    return output

In [ ]:
def analyze_user_reviews(reviews_list):
  delimiter = "#####"

  assistant_response_sample = {
      "Ambience": "excellent",
		  "Parking": "No",
		  "Suggested Food": [
        'Pumpkin Halwa',
			  'kulchas',
			  'Cheese chilli paratha',
			  'Bhutta palak methi curry',
			  'Lemon Chicken',
			  'Egg fried rice',
			  '"Alice in Junoon"',
			  'soya chaap'
      ]
	}

  review_summary = {
		"Ambience":None,
		"Parking":None,
		"Suggested Food":None
	}

  prompt= f"""
	You are a very proficient restaurant review analyzer whose job is to extract the key features from user reviews.
	To analyze the reviews of each restaurant, perform the following steps:
	Step1: Extract the reviews list of the restaurant from the reviews description {reviews_list}\
	Step2: Store the extracted features in {review_summary} \
	Step3: Fill in the values of each features in {review_summary} based on collabarative review from users and applying the below rules:
	{delimiter}
	Ambience - Choose from [family-friendly, romantic, kid-friendly, friends-hangout, bad, excellent]
	Parking availability - Choose from [Yes,No]
	Suggested Food - Can be a list of food items suggested by different users
	{delimiter}

  Output:
	{delimiter}
	Output the python dictionary in the below format with None values if the information is not available from user reviews. Exclude the Keys which has 'None' value
	{review_summary}
	{delimiter}

  Examples:
	{delimiter}
	User: Here is the reviews for a specific restaurant
	[('Rated 4.0', 'RATED A beautiful place to dine in.The interiors take you back to the Mughal era. The lightings are just perfect.We went there on the occasion of Christmas and so they had only limited items available. But the taste and service was not compromised at all.The only complaint is that the breads could have been better.Would surely like to come here again.'), ('Rated 4.0', 'RATED I was here for dinner with my family on a weekday. The restaurant was completely empty. Ambience is good with some good old hindi music. Seating arrangement are good too. We ordered masala papad, panner and baby corn starters, lemon and corrionder soup, butter roti, olive and chilli paratha. Food was fresh and good, service is good too. Good for family hangout.Cheers'), ('Rated 2.0', 'RATED Its a restaurant near to Banashankari BDA. Me along with few of my office friends visited to have buffet but unfortunately they only provide veg buffet. On inquiring they said this place is mostly visited by vegetarians. Anyways we ordered ala carte items which took ages to come. Food was ok ok. Definitely not visiting anymore.'), ('Rated 4.0', 'RATED We went here on a weekend and one of us had the buffet while two of us took Ala Carte. Firstly the ambience and service of this place is great! The buffet had a lot of items and the good was good. We had a Pumpkin Halwa intm the dessert which was amazing. Must try! The kulchas are great here. Cheers!'), ('Rated 5.0', 'RATED The best thing about the place is its ambiance. Second best thing was yummy ? food. We try buffet and buffet food was not disappointed us.Test ?. ?? ?? ?? ?? ??Quality ?. ??????????.Service: Staff was very professional and friendly.Overall experience was excellent.subirmajumder85.wixsite.com'), ('Rated 5.0', 'RATED Great food and pleasant ambience. Expensive but Coll place to chill and relax......Service is really very very good and friendly staff...Food : 5/5Service : 5/5Ambience :5/5Overall :5/5'), ('Rated 4.0', 'RATED Good ambience with tasty food.Cheese chilli paratha with Bhutta palak methi curry is a good combo.Lemon Chicken in the starters is a must try item.Egg fried rice was also quite tasty.In the mocktails, recommend "Alice in Junoon". Do not miss it.'), ('Rated 4.0', 'RATED You cant go wrong with Jalsa. Never been a fan of their buffet and thus always order alacarte. Service at times can be on the slower side but food is worth the wait.'), ('Rated 5.0', 'RATED Overdelighted by the service and food provided at this place. A royal and ethnic atmosphere builds a strong essence of being in India and also the quality and taste of food is truly authentic. I would totally recommend to visit this place once.'), ('Rated 4.0', 'RATED The place is nice and comfortable. Food wise all jalea outlets maintain a good standard. The soya chaap was a standout dish. Clearly one of trademark dish as per me and a must try.The only concern is the parking. It very congested and limited to just 5cars. The basement parking is very steep and makes it cumbersome'), ('Rated 4.0', 'RATED The place is nice and comfortable. Food wise all jalea outlets maintain a good standard. The soya chaap was a standout dish. Clearly one of trademark dish as per me and a must try.The only concern is the parking. It very congested and limited to just 5cars. The basement parking is very steep and makes it cumbersome'), ('Rated 4.0', 'RATED The place is nice and comfortable. Food wise all jalea outlets maintain a good standard. The soya chaap was a standout dish. Clearly one of trademark dish as per me and a must try.The only concern is the parking. It very congested and limited to just 5cars. The basement parking is very steep and makes it cumbersome')]
  Assitant: {assistant_response_sample}
	"""
  input = f"""Follow the above instructions step-by-step and output the dictionary in JSON format {review_summary} for the following user review {reviews_list}."""
  messages=[{"role": "system", "content":prompt },{"role": "user","content":input}]

  response = get_chat_completions(messages, json_format = True)

  return response

Test the above code with small dataset

In [ ]:
zomato_df_test = zomato_df.head(2).copy()

In [ ]:
zomato_df_test

,address,name,online_order,book_table,rate,votes,location,rest_type,cuisines,cost,reviews_list,menu_item,type,city
0,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,Yes,Yes,4.1,775,Banashankari,Casual Dining,"North Indian, Mughlai, Chinese",800.0,"[('Rated 4.0', 'RATED A beautiful place to di...",[],Buffet,Banashankari
1,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,Yes,No,4.1,787,Banashankari,Casual Dining,"Chinese, North Indian, Thai",800.0,"[('Rated 4.0', 'RATED Had been here for dinne...",[],Buffet,Banashankari


In [ ]:
## Create a new column "review_summary" that contains the collabarative summary of user reviews on the restaurant
zomato_df_test['review_summary'] = zomato_df_test['reviews_list'].apply(lambda x: analyze_user_reviews(x))

In [ ]:
zomato_df_test.head()

,address,name,online_order,book_table,rate,votes,location,rest_type,cuisines,cost,reviews_list,menu_item,type,city,review_summary
0,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,Yes,Yes,4.1,775,Banashankari,Casual Dining,"North Indian, Mughlai, Chinese",800.0,"[('Rated 4.0', 'RATED A beautiful place to di...",[],Buffet,Banashankari,"{'Ambience': 'excellent', 'Parking': 'No', 'Su..."
1,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,Yes,No,4.1,787,Banashankari,Casual Dining,"Chinese, North Indian, Thai",800.0,"[('Rated 4.0', 'RATED Had been here for dinne...",[],Buffet,Banashankari,"{'Ambience': 'family-friendly', 'Parking': 'No..."


In [ ]:
pd.set_option('display.max_colwidth', None)
zomato_df_test['review_summary']

,review_summary
0,"{'Ambience': 'excellent', 'Parking': 'No', 'Suggested Food': ['masala papad', 'panner', 'baby corn starters', 'lemon and coriander soup', 'butter roti', 'olive and chilli paratha', 'Pumpkin Halwa', 'kulchas', 'Cheese chilli paratha', 'Bhutta palak methi curry', 'Lemon Chicken', 'Egg fried rice', '""Alice in Junoon""', 'soya chaap']}"
1,"{'Ambience': 'family-friendly', 'Parking': 'No', 'Suggested Food': ['Chicken biriyani', 'Mutton biriyani', 'Thom yum Thai soup', 'panner curry', 'paneer uttar dakshin', 'paneer kurchan', 'Gobi hara pyaz', 'mix veg', 'Lassi', 'Spice elephant soup', 'Lasooni fish tikka']}"


Data is filled by OpenAI API to review_summary column as expected. Now fill in the details to our actual dataset.

In [ ]:
## Create a new column "review_summary" that contains the collabarative summary of user reviews on the restaurant
zomato_df['review_summary'] = zomato_df['reviews_list'].apply(lambda x: analyze_user_reviews(x))

KeyboardInterrupt: 

Since the actual dataset is too huge, lets take a subset of the same as our data.

In [ ]:
zomato_df_new = zomato_df.head(1000).copy()

In [ ]:
zomato_df.shape

(40708, 14)

In [ ]:
zomato_df_new.shape

(1000, 14)

In [ ]:
## Create a new column "review_summary" that contains the collabarative summary of user reviews on the restaurant
zomato_df_new['review_summary'] = zomato_df_new['reviews_list'].apply(lambda x: analyze_user_reviews(x))

In [ ]:
pd.reset_option('^display.', silent=True)
zomato_df_new.head(2)

,address,name,online_order,book_table,rate,votes,location,rest_type,cuisines,cost,reviews_list,menu_item,type,city,review_summary
0,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,Yes,Yes,4.1,775,Banashankari,Casual Dining,"North Indian, Mughlai, Chinese",800.0,"[('Rated 4.0', 'RATED A beautiful place to di...",[],Buffet,Banashankari,"{'Ambience': 'excellent', 'Parking': 'No', 'Su..."
1,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,Yes,No,4.1,787,Banashankari,Casual Dining,"Chinese, North Indian, Thai",800.0,"[('Rated 4.0', 'RATED Had been here for dinne...",[],Buffet,Banashankari,"{'Ambience': 'family-friendly', 'Parking': 'No..."


In [ ]:
pd.set_option('display.max_colwidth', None)
zomato_df_new['review_summary'].head(5)

,review_summary
0,"{'Ambience': 'excellent', 'Parking': 'No', 'Suggested Food': ['masala papad', 'panner', 'baby corn starters', 'lemon and coriander soup', 'butter roti', 'olive and chilli paratha', 'Pumpkin Halwa', 'kulchas', 'Cheese chilli paratha', 'Bhutta palak methi curry', 'Lemon Chicken', 'Egg fried rice', '""Alice in Junoon""', 'soya chaap']}"
1,"{'Ambience': 'family-friendly', 'Parking': 'No', 'Suggested Food': ['Chicken biriyani', 'Mutton biriyani', 'Thom yum Thai soup', 'panner curry', 'paneer uttar dakshin', 'paneer kurchan', 'Gobi hara pyaz', 'mix veg', 'Lassi', 'Spice elephant soup', 'Lasooni fish tikka']}"
2,"{'Ambience': 'bad', 'Suggested Food': ['Desserts', 'Nutella churros', 'Thai green curry', 'Churros', 'Pizza', 'Hot chocolate', 'Nachos', 'Pink pasta', 'Caramel pudding']}"
3,"{'Ambience': 'average', 'Parking': 'No', 'Suggested Food': ['masala dosa', 'holige', 'pineapple pickle', 'Payasam', 'Kosambari', 'appekai saru', 'masala dosa', 'pulka', 'pulav', 'veg rice bath', 'kai holige', 'mango soup']}"
4,"{'Ambience': 'family-friendly', 'Suggested Food': ['Jaljeera', 'buttermilk', 'chat papdi', 'bhajiya', 'dosa', 'pav bhaji', 'noodles', 'curries', 'kulcha', 'roti', 'Jalebies', 'gulab jamuns']}"


Store the updated dataframe to a CSV file, so that if the server connection is lost, we need not hit the OpenAI API again to get review_summary, rather load the data from CSV file

In [ ]:
zomato_df_new.to_csv("../data/updated_zomato_data.csv",index=False,header = True)

##### 3.2 Copy only the required columns to the new dataframe so as to reduce the number of input tokens

In [ ]:
zomato_df_new = zomato_df_new[['name','address','online_order','book_table','rate','location','cuisines','cost','review_summary']].copy()

In [ ]:
pd.reset_option('^display.', silent=True)
zomato_df_new.head(2)

,name,address,online_order,book_table,rate,location,cuisines,cost,review_summary
0,Jalsa,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Yes,Yes,4.1,Banashankari,"North Indian, Mughlai, Chinese",800.0,"{'Ambience': 'excellent', 'Parking': 'No', 'Su..."
1,Spice Elephant,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Yes,No,4.1,Banashankari,"Chinese, North Indian, Thai",800.0,"{'Ambience': 'family-friendly', 'Parking': 'No..."


In [ ]:
review_summary_df = pd.json_normalize(zomato_df_new['review_summary'])

In [ ]:
zomato_df_updated = pd.concat([zomato_df_new, review_summary_df], axis=1, join='inner')

In [ ]:
zomato_df_updated = zomato_df_updated.drop(['cuisines', 'review_summary','online_order','book_table'], axis=1)

In [ ]:
zomato_df_updated.head()

,name,address,rate,location,cost,Ambience,Parking,Suggested Food
0,Jalsa,"942, 21st Main Road, 2nd Stage, Banashankari, ...",4.1,Banashankari,800.0,excellent,No,"[masala papad, panner, baby corn starters, lem..."
1,Spice Elephant,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",4.1,Banashankari,800.0,family-friendly,No,"[Chicken biriyani, Mutton biriyani, Thom yum T..."
2,San Churro Cafe,"1112, Next to KIMS Medical College, 17th Cross...",3.8,Banashankari,800.0,bad,NaN,"[Desserts, Nutella churros, Thai green curry, ..."
3,Addhuri Udupi Bhojana,"1st Floor, Annakuteera, 3rd Stage, Banashankar...",3.7,Banashankari,300.0,average,No,"[masala dosa, holige, pineapple pickle, Payasa..."
4,Grand Village,"10, 3rd Floor, Lakshmi Associates, Gandhi Baza...",3.8,Basavanagudi,600.0,family-friendly,NaN,"[Jaljeera, buttermilk, chat papdi, bhajiya, do..."


In [63]:
zomato_df_updated.fillna('', inplace=True)

In [ ]:
zomato_df_updated['Suggested Food'] = [','.join(map(str, l)) for l in zomato_df_updated['Suggested Food']]

In [ ]:
zomato_df_updated.head()

,name,address,rate,location,cost,Ambience,Parking,Suggested Food
0,Jalsa,"942, 21st Main Road, 2nd Stage, Banashankari, ...",4.1,Banashankari,800.0,excellent,No,"masala papad,panner,baby corn starters,lemon a..."
1,Spice Elephant,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",4.1,Banashankari,800.0,family-friendly,No,"Chicken biriyani,Mutton biriyani,Thom yum Thai..."
2,San Churro Cafe,"1112, Next to KIMS Medical College, 17th Cross...",3.8,Banashankari,800.0,bad,,"Desserts,Nutella churros,Thai green curry,Chur..."
3,Addhuri Udupi Bhojana,"1st Floor, Annakuteera, 3rd Stage, Banashankar...",3.7,Banashankari,300.0,average,No,"masala dosa,holige,pineapple pickle,Payasam,Ko..."
4,Grand Village,"10, 3rd Floor, Lakshmi Associates, Gandhi Baza...",3.8,Basavanagudi,600.0,family-friendly,,"Jaljeera,buttermilk,chat papdi,bhajiya,dosa,pa..."


### 4. Using GeoCoding API to get the list of areas in the CSV file with their coordinate values

In [ ]:
zomato_df_updated['location'].value_counts()

,count
location,
Banashankari,392
Basavanagudi,160
Bannerghatta Road,108
JP Nagar,94
Jayanagar,93
BTM,66
Kumaraswamy Layout,50
Mysore Road,15
Uttarahalli,9


In [44]:
def get_lat_long(location_name):

    location_name = location_name+", Bangalore, India"
    # Initialize Nominatim API
    geolocator = Nominatim(user_agent="RestaurantAssistAI")

    # Get location data
    location = geolocator.geocode(location_name)

    if location:
        # Return the latitude and longitude
        return (location.latitude, location.longitude)
    else:
        return None

In [8]:
# Example usage
location_name = "Jayanagar"
coordinates = get_lat_long(location_name)
print(coordinates[0])
print(coordinates[1])

12.9418488
77.5868976


In [46]:
#Get Unique locations
unique_locations = zomato_df_updated['location'].unique()
location_df = pd.DataFrame(unique_locations, columns=['location'])

In [47]:
location_df.shape

(14, 1)

In [24]:
location_df.head(2)

,location
0,Banashankari
1,Basavanagudi


Fill in the lat long values for each of these locations

In [48]:
location_df['coordinates'] = location_df['location'].apply(lambda x: get_lat_long(x))

In [49]:
location_df.head()

,location,coordinates
0,Banashankari,"(12.9393328, 77.5539819)"
1,Basavanagudi,"(12.9417261, 77.5755021)"
2,Mysore Road,"(12.9597674, 77.556145)"
3,Jayanagar,"(12.9399039, 77.5826382)"
4,Kumaraswamy Layout,"(12.9067683, 77.5595021)"


Store these details to a CSV file, so that even if the server connection is lost, we need not run the API to get these details

In [134]:
zomato_df_updated.to_csv("../data/zomato_df_updated.csv",index=False,header = True)
location_df.to_csv("../data/location_df.csv",index=False,header = True)

### 5. Get the extracted data from CSV files

In [50]:
def get_input_data_from_csv():
  zomato_df_updated = pd.read_csv("../data/zomato_df_updated.csv")
  #location_df_updated = pd.read_csv("/content/drive/MyDrive/Project_data/FoodRecommendationAI/location_df.csv")
  location_df_updated = location_df.copy()
  return zomato_df_updated,location_df_updated

In [51]:
zomato_df_updated,location_df_updated = get_input_data_from_csv()

In [52]:
zomato_df_updated.fillna('Not Known', inplace=True)

In [53]:
zomato_df_updated['Parking'].value_counts()

,count
Parking,
Not Known,872
No,75
Yes,53


In [14]:
pd.reset_option('^display.', silent=True)

In [15]:
zomato_df_updated.head(2)

,name,address,rate,location,cost,Ambience,Parking,Suggested Food
0,Jalsa,"942, 21st Main Road, 2nd Stage, Banashankari, ...",4.1,Banashankari,800.0,excellent,No,"masala papad,panner,baby corn starters,lemon a..."
1,Spice Elephant,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",4.1,Banashankari,800.0,family-friendly,No,"Chicken biriyani,Mutton biriyani,Thom yum Thai..."


In [16]:
zomato_df_updated.shape

(1000, 8)

In [54]:
location_df_updated.head(2)

,location,coordinates
0,Banashankari,"(12.9393328, 77.5539819)"
1,Basavanagudi,"(12.9417261, 77.5755021)"


In [18]:
location_df_updated.shape

(14, 2)

### 6. Data extraction Layer

Here we are going to use LangChain with OpenAI API capability to extract the information from pandas dataframe 'zomato_df_updated' based on the user requirement in natural language.

##### 6.1 Helper function to get the list of nearby areas for the given input location

Note that it tries to find the near by area for the given location from "location_df" dataframe, as the intent here is if the location entered by user is not found in our database, we need to take the nearest area which is there as part of the database and then recommend the restaurants in that area. By default 3km is taken as standard for selecting the nearby area.

In [55]:
def find_areas_within_radius(user_lat, user_lon, df, radius_km=3):
    result = []
    user_location = (user_lat, user_lon)

    for index, row in df.iterrows():
        area_location = row['coordinates']
        # Calculate distance between user location and area location
        dist = distance(user_location, area_location).km
        # If distance is within the specified radius, add area to result
        if dist <= radius_km:
            result.append(row['location'])

    return result

In [56]:
def getAreaList(user_input_area,radius_km=3):
  arealist=[]
  #Check whether user input is there as part of the master list
  if location_df.apply(lambda row: user_input_area in row.values, axis=1).any():
    arealist.append(user_input_area)
    return arealist
  else:
    #Get the lat long of the input area
    coordinates = get_lat_long(user_input_area)
    # Find areas within 3 km radius
    areas_within_radius = find_areas_within_radius(coordinates[0], coordinates[1], location_df_updated)
    return areas_within_radius

In [57]:
#Example of how this works
#Below is the lat long coordinates of 'Gowdanapalya'
user_lat = 12.9093
user_lon = 77.5580

# Find areas within 3 km radius
areas_within_radius = find_areas_within_radius(user_lat, user_lon, location_df_updated)

# Output the result
print("Areas within 3 km radius:", areas_within_radius)

Areas within 3 km radius: ['Kumaraswamy Layout', 'Uttarahalli']


##### 6.2 Using LangChain OpenAI API to extract the data from pandas dataframe which converts natural language to Python dataframe query

In [107]:
def table_to_json_conv(data):
  # Split data into lines and remove any extra lines
  lines = data.strip().split('\n')

  # Extract the header and the rows
  header = lines[0].strip().split('|')[1:-1]  # Extract column headers
  rows = lines[0:]  # Skip the separator line

  # Initialize a list to hold the parsed data
  restaurants = []

  # Iterate over each row to extract and parse values
  for row in rows:
    # Split the row based on the table delimiters (|), and remove extra spaces
    columns = row.strip().split('|')[1:-1]
    #columns = [col.strip() for col in columns]  # Clean up whitespace
    if len(columns)==0:
      print("Empty column returned.")
      break

    # Parse the Suggested Food list (comma-separated)
    suggested_food = [food.strip() for food in columns[6].rstrip().split(',')]

    # Create a dictionary for the restaurant entry
    restaurant = {
        "Name": columns[0].rstrip(),
        "Address": columns[1].rstrip(),
        "Rate": columns[2].rstrip(),
        "Cost": columns[3].rstrip(),
        "Ambience": columns[4].rstrip(),
        "Parking": columns[5].rstrip() if columns[5] else "No",  # Handle missing parking info
        "Suggested Food": suggested_food
    }

    # Append to the restaurants list
    restaurants.append(restaurant)

  # Convert the list of restaurants to JSON format
  result_json = json.dumps(restaurants, indent=4)
  return result_json

In [130]:
agent = create_pandas_dataframe_agent(
    ChatOpenAI(temperature=0, model="gpt-4o-mini"),
    zomato_df_updated,
    verbose=False,
    agent_type=AgentType.OPENAI_FUNCTIONS,
    allow_dangerous_code=True
)

In [131]:
def extract_data_from_dataframe(user_location, user_query):


  prompt = f"""
   Get the name, address, rate, cost, ambience, parking and suggested food.

   Guidelines:
   ####
   Return the result in table format only. Do not include header or column names.
   Do not include row number in the table.
   If no data found, return the String 'No Data Found'
   ####

   You will be heavily penalised if the above guidelines is not followed

   Example:
   ####
   User: "Looking for top 5 Veg restaurants in Banashankari."
   Assistant:
   | Taaza Thindi    | 115, 100 Feet Ring Road, Kathriguppe, Banashankari, Bangalore     |    4.7 |    100 | excellent | Not Known | khaara bath,uddina vada,Masala Dosa,Kesari Bath                                               |
   | Onesta          | 2469, 3rd Floor, 24th Cross, Opposite BDA Complex, Banashankari   |    4.6 |    600 | excellent | Not Known | starters,pizzas,pasta,tiramisu shots,mozzarella cheese garlic bread                               |
   | Taaza Thindi    | 115, 100 Feet Ring Road, Kathriguppe, Banashankari, Bangalore     |    4.7 |    100 | excellent | Not Known | khaara bath,uddina vada,Masala Dosa,Kesari Bath                                               |
   | Onesta          | 2469, 3rd Floor, 24th Cross, Opposite BDA Complex, Banashankari   |    4.6 |    600 | excellent | Not Known | starters,pizzas,pasta,tiramisu shots,mozzarella cheese garlic bread                               |
   | Poonam Sweets   | 43, BDA Complex, 2nd Stage, Banashankari, Bangalore               |    4.4 |    150 | bad      | Not Known | samosa,kachori,jalebi,Dhokla,Basundi,Badam Milk                                                  |
   ####
  """
  user_query = user_query+prompt

  output = agent.run(user_query)

  if output == 'No Data Found':
    #Get the nearest location to the given area
    new_user_location = getAreaList(user_location)
    new_user_location = ', '.join(new_user_location)
    print("No records found for location entered by user. Checking for the nearest location:",new_user_location)

    user_query = user_query.replace(user_location, new_user_location)
    output = agent.run(user_query)
    print(output)
    result_json = table_to_json_conv(output)

  else:
    # Converting the result to JSON format
    result_json = table_to_json_conv(output)

  return result_json

In [109]:
#Example Usage for the location which is present in database
user_location = "Banashankari"
user_query = "Looking for unique top 5 Veg restaurants in Banashankari."
response = extract_data_from_dataframe(user_location,user_query)
print("response:\n", response)



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "unique_veg_restaurants = df[(df['location'] == 'Banashankari') & (df['Suggested Food'].str.contains('paneer|dosa|veg|salad|dal|sabzi|bhaji|bhajiya|pulao|biryani|chole|chutney|idli|paratha|pav|kadhi|kheer|puri|samosa|kachori|dhokla|basundi|bhindi|gobi|methi|palak|mixed veg|vegetable|vegetarian', case=False))].drop_duplicates(subset=['name']).head(5) \nresult = unique_veg_restaurants[['name', 'address', 'rate', 'cost', 'Ambience', 'Parking', 'Suggested Food']].to_string(index=False, header=False) \nresult if not unique_veg_restaurants.empty else 'No Data Found'"}`


                Jalsa                                              942, 21st Main Road, 2nd Stage, Banashankari, Bangalore 4.1 800.0       excellent        No                                                                                        masala papad,panner,baby corn starters,lemon and coriander soup,butter roti,olive and chilli para

In [110]:
#Example Usage for the location which is not present in database
user_location = "Gowdanapalya"
user_query = "Looking for unique top 5 restaurants in Gowdanapalya."
response = extract_data_from_dataframe(user_location,user_query)
print("response:\n", response)



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "unique_restaurants = df[df['location'] == 'Gowdanapalya'].drop_duplicates(subset=['name']).head(5); unique_restaurants[['name', 'address', 'rate', 'cost', 'Ambience', 'Parking', 'Suggested Food']].to_string(index=False, header=False) if not unique_restaurants.empty else 'No Data Found'"}`


No Data FoundNo Data Found

> Finished chain.
No records found for location entered by user. Checking for the nearest location: Kumaraswamy Layout, Uttarahalli


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "df[(df['location'] == 'Kumaraswamy Layout') | (df['location'] == 'Uttarahalli')].drop_duplicates(subset=['name']).head(5)"}`


               name                                            address  rate  \
46   Kitchen Garden  1750, 14th Main, Police Station Road, Kumarasw...   3.6   
47           Recipe  1621, 1st Floor, 50 Feet Main Road, Kumaraswam...   4.0   
49      T

### 7. Moderation Layer

##### `moderation_check()`:
 This checks if the user's or the assistant's message is inappropriate. If any of these is inappropriate, you can add a break statement to end the conversation.

In [81]:
# Define a function called moderation_check that takes user_input as a parameter.

def moderation_check(user_input):
    # Call the OpenAI API to perform moderation on the user's input.
    response = openai.moderations.create(input=user_input)

    # Extract the moderation result from the API response.
    moderation_output = response.results[0].flagged
    # Check if the input was flagged by the moderation system.
    if response.results[0].flagged == True:
        # If flagged, return "Flagged"
        return "Flagged"
    else:
        # If not flagged, return "Not Flagged"
        return "Not Flagged"

In [63]:
#Example check
moderation_check("I want to kill them.")

'Flagged'

In [64]:
moderation_check("Get me the list of restaurants in Uttarahalli.")

'Not Flagged'

### 8. User Input Layer and Intent Confirmation Layer

##### 8.1 Chat Completions API from OpenAI

In [82]:
@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(6))
def get_chat_completions(input, json_format = False):
    MODEL = 'gpt-4o-mini'

    system_message_json_output = """<<. Return output in JSON format to the key output.>>"""

    # If the output is required to be in JSON format
    if json_format == True:
        # Append the input prompt to include JSON response as specified by OpenAI
        input[0]['content'] += system_message_json_output

        # JSON return type specified
        chat_completion_json = openai.chat.completions.create(
            model = MODEL,
            messages = input,
            response_format = { "type": "json_object"},
            seed = 1234)

        output = json.loads(chat_completion_json.choices[0].message.content)

    # No JSON return type specified
    else:
        chat_completion = openai.chat.completions.create(
            model = MODEL,
            messages = input,
            seed = 2345)

        output = chat_completion.choices[0].message.content

    return output

##### 8.2 Helper function dictionary_present()

This function checks if the final understanding of user's profile is returned by the chatbot is a Python dictionary or not.

In [83]:
def dictionary_present(response):
    delimiter = "####"

    user_req = {'location': 'Banashankari',
                'prompt': 'Looking for Italian or Chinese pure vegetarian restaurants in Banashankari, with a family-friendly Ambience and no specific budget constraints.'
                }

    prompt = f"""You are a python expert. You are provided an input.
            You have to check if there is a python dictionary present in the string.
            It will have the following format {user_req}.
            Your task is to just extract the relevant values from the input and return only the python dictionary in JSON format.
            The output should match the format as {user_req}.

            {delimiter}
            Make sure that the value of location is also present in the user input. ###
            The output should contain the exact keys and values as present in the input.
            Ensure the keys and values are in the given format:
            {{
            'location': 'string',
            'prompt':'string'
            }}
            Here are some sample input output pairs for better understanding:
            {delimiter}
            input 1: <<<location: Banashankari - prompt: Looking for chinese restaurants in Banashankari>>>
            output 1: <<<{{'location': 'Banashankari', 'prompt': 'Looking for chinese restaurants in Banashankari'}}>>>

            input 2: <<<Thank you for providing all the details! Here's the final prompt:
            ```json
                'location': 'Banashankari', 'prompt': 'Looking for chinese restaurants in Banashankari'
            ```>>>
            output 2: <<<{{'location': 'Banashankari', 'prompt': 'Looking for chinese restaurants in Banashankari'}}>>>
            {delimiter}
            """
    messages = [{"role": "system", "content":prompt },
                {"role": "user", "content":f"""Here is the user input: {response}""" }]

    confirmation = get_chat_completions(messages, json_format = True)

    return confirmation

In [92]:
def initialize_conversation():

  '''
    Returns a list [{"role": "system", "content": system_message}]
  '''

  delimiter = "#####"

  sample_response = {
		"location":"_",
		"prompt":"_"
	}

  example_user_response = {
      "location":"Banashankari",
      "prompt":"Looking for Italian or Chinese pure vegetarian restaurants in Banashankari, with a family-friendly Ambience and no specific budget constraints."
  }

  prompt= f"""

  You are very proficient and have vast experience in recommending suitable and most appropriate restaurants based on individuals preference and needs.
  You are expected to interact with user and get the below set of requirements or user preferences
  Start with a warm greeting message and your final objective is to populate the following features and frame a well defined, detailed sentence captuing all these details
  Include the text 'Top 5 distinct restaurants' in the response string.

  {delimiter}
  Location
  Cuisine Preferences
  Dietary Restrictions
  Ambience
  Budget
  Additional Preferences
  {delimiter}

  Instructions for filling the values:
  {delimiter}
  Ensure that the values for the features follow these rules:
  Location - location in which user is trying to find the restaurants
  Cuisine Preferences - Any specific cuisine the user is looking for. Choose from [Indian, Italian, Asian, Continental, Chinese, Desserts]
  Dietary Restrictions - Choose from [Veg, Non-Veg, Vegan, Jain]
  Ambience - Choose from [family-friendly, romantic, kid-friendly, friends-hangout]
  Budget - In Rupees and would be a budget for 2 people. The value for Budget should be a numerical value
  Additional Preferences - Any requirement from user that does not fall under above defined categories
  {delimiter}

  Guidelines
  {delimiter}
  Output should be a passive sentence without referring or addressing the user, so that it can be used as a prompt for other API call.
  Do not bombard all questions at once. Be smart enough to ask relevant questions in a conversational, non-intrusive manner.
  The value for Budget should be a numerical value extracted from the user's response.
  Include the text 'Top 5 restaurants' in the response string.
  {delimiter}

  Output:
	{delimiter}
	Output the python dictionary in the below format.
  {sample_response}
  location - should correspond to the location entered by the user
  prompt - Output should be a passive sentence without referring or addressing the user, so that it can be used as a prompt for other API call.
	{delimiter}

  Example:
  {delimiter}
  Assistant: "Are you looking for any specific cuisine like Indian, Italian, Asian, Continental, Chinese or Desserts?"
  User: "Italian or chinese"
  Assistant: "Noted! Do you have any dietary restrictions, such as Veg, Non-Veg, Vegan, or Jain?"
  User: "Pure veg"
  Assistant: "Understood! For the Ambience, would you prefer something family-friendly, romantic, kid-friendly or friends-hangout?"
  User: "family-friendly"
  Assistant: "Thank you for the details! Here's the final prompt:"
  {example_user_response}
  {delimiter}

  """

  conversation=[{"role": "system", "content":prompt }]

  return conversation

In [93]:
debug_conversation = initialize_conversation()

In [94]:
print(debug_conversation[0]['content'])



  You are very proficient and have vast experience in recommending suitable and most appropriate restaurants based on individuals preference and needs.
  You are expected to interact with user and get the below set of requirements or user preferences
  Start with a warm greeting message and your final objective is to populate the following features and frame a well defined, detailed sentence captuing all these details
  Include the text 'Top 5 distinct restaurants' in the response string.

  #####
  Location
  Cuisine Preferences
  Dietary Restrictions
  Ambience
  Budget
  Additional Preferences
  #####

  Instructions for filling the values:
  #####
  Ensure that the values for the features follow these rules:
  Location - location in which user is trying to find the restaurants
  Cuisine Preferences - Any specific cuisine the user is looking for. Choose from [Indian, Italian, Asian, Continental, Chinese, Desserts]
  Dietary Restrictions - Choose from [Veg, Non-Veg, Vegan, Jain]
  

`intent_confirmation_layer()`:

This function takes the assistant's response and evaluates if the chatbot has captured the user's requirement clearly. Specifically, this checks if the following properties for the user has been captured or not
   - location
   - prompt

In [88]:
def intent_confirmation_layer(response_assistant):

    delimiter = "####"

    prompt = f"""
    You are a senior evaluator who has an eye for detail.The input text will contain a user requirement captured through 2 keys.
    You are provided an input. You need to evaluate if the input text has the following keys:
    {{
    'location': 'values',
    'prompt':'values'
    }}
    Only output a one-word string in JSON format at the key 'result' - Yes/No.
    Thought 1 - Output a string 'Yes' if the values are correctly filled for all keys, otherwise output 'No'.
    Thought 2 - If the answer is No, mention the reason in the key 'reason'.
    THought 3 - Think carefully before the answering.
    """
    messages=[{"role": "system", "content":prompt },
              {"role": "user", "content":f"""Here is the input: {response_assistant}""" }]

    response = openai.chat.completions.create(
                                    model="gpt-3.5-turbo",
                                    messages = messages,
                                    response_format={ "type": "json_object" },
                                    seed = 1234
                                    # n = 5
                                    )
    json_output = json.loads(response.choices[0].message.content)

    return json_output

In [89]:
#Example Usage
example_user_response = {
      "location":"Banashankari",
      "prompt":"Looking for Italian or Chinese pure vegetarian restaurants in Banashankari, with a family-friendly ambiance and no specific budget constraints."
  }

confirmation_response = intent_confirmation_layer(example_user_response)
print(confirmation_response)

{'result': 'Yes'}


In [90]:
#Example Usage
example_user_response = {
     "prompt":"Looking for Italian or Chinese pure vegetarian restaurants in Banashankari, with a family-friendly ambiance and no specific budget constraints."
  }

confirmation_response = intent_confirmation_layer(example_user_response)
print(confirmation_response)

{'result': 'No', 'reason': "Missing 'location' key"}


### 9. Restaurant recommendation layer

It takes the list of top 5 restaurants returned from data extraction layer (using langchain and OpenAI) as input and based on that list and the user input, it provides recommendation on one of the restaurant that matches the user input.
It has the following steps:
1. Initialize the conversation for recommendation.
2. Generate the recommendations and display in a presentable format.

In [112]:
def initialize_conv_reco(restaurants, user_requirement):
    system_message = f"""
    You are an intelligent restaurant review expert and you are tasked with the objective to \
    solve the user queries about top restaurant with the requirements from the catalogue in the user message \
    You should keep the user profile in mind while answering the questions.\

    Pick top 1 restaurant from the list that matches exactly with the user requirements
    and also has better qualities that any user would expect.
    Provide a summarized recommendation as output that can truely influence the user decision in picking the restaurant.

    """
    user_message = f""" These are the list of restaurants: {restaurants}"""
    conversation = [{"role": "system", "content": system_message },
                    {"role":"user","content":user_message}]

    conversation.append({"role": "user", "content": user_requirement})
    return conversation

### 10. Dialouge Management System

Bringing everything together, we create a diagloue_mgmt_system() function that contains the logic of how the different layers would interact with each other. This will be the function that we'll call to initiate the chatbot

In [115]:
def dialogue_mgmt_system():
    conversation = initialize_conversation()

    #display("conversation:",conversation)
    introduction = get_chat_completions(conversation)

    display(introduction + '\n')

    user_input = ''
    top_5_restaurants = ''

    while(user_input != "exit"):
      user_input = input("")

      moderation = moderation_check(user_input)
      if moderation == 'Flagged':
         display("Sorry, this message has been flagged. Please restart your conversation.")
         break

      if top_5_restaurants == '':

        conversation.append({"role": "user", "content": user_input})
        response_assistant = get_chat_completions(conversation)

        moderation = moderation_check(response_assistant)
        if moderation == 'Flagged':
           display("Sorry, this message has been flagged. Please restart your conversation.")
           break

        confirmation = intent_confirmation_layer(response_assistant)

        if "No" in confirmation.get('result'):
          conversation.append({"role": "assistant", "content": str(response_assistant)})
          print("\n" + str(response_assistant) + "\n")

        else:
          print("\n" + str(response_assistant) + "\n")
          response = dictionary_present(response_assistant)
          response_str = response['output']
          response_json = json.loads(response_str.replace("'","\""))

          print("Thank you for providing all the information. Kindly wait, while I fetch the restaurant details: \n")
          top_5_restaurants = extract_data_from_dataframe(response_json['location'],response_json['prompt'])

          moderation = moderation_check(top_5_restaurants)
          if moderation == 'Flagged':
            display("Sorry, this message has been flagged. Please restart your conversation.")
            break
          else:
            print("------------------ Top Restaurants as per your requirement -------------------")
            print(top_5_restaurants)

          restaurant_reco = initialize_conv_reco(top_5_restaurants,response_json['prompt'])
          recommendation = get_chat_completions(restaurant_reco)

          moderation = moderation_check(recommendation)
          if moderation == 'Flagged':
            display("Sorry, this message has been flagged. Please restart your conversation.")
            break

          restaurant_reco.append({"role": "assistant", "content": str(recommendation)})
          print("-------------------- Recommendation --------------------")
          print('\n' + str(recommendation) + '\n')

      else:
         restaurant_reco.append({"role": "user", "content": user_input})

         response_asst_reco = get_chat_completions(restaurant_reco)

         moderation = moderation_check(response_asst_reco)
         if moderation == 'Flagged':
          print("Sorry, this message has been flagged. Please restart your conversation.")
          break

         print('\n' + response_asst_reco + '\n')
         conversation.append({"role": "assistant", "content": response_asst_reco})

### 11. Testing the code

##### Case1: For the Location which is present in database and minimal user requirement




In [119]:
dialogue_mgmt_system()

"Hello! I'm here to help you find the perfect dining experience. To get started, could you please tell me the location where you're looking for restaurants?\n"

Banashankari

Thank you for sharing the location! Next, are you looking for any specific cuisine like Indian, Italian, Asian, Continental, Chinese, or Desserts?

Indian

Noted! Do you have any dietary restrictions, such as Veg, Non-Veg, Vegan, or Jain?

no

Understood! For the ambience, would you prefer something family-friendly, romantic, kid-friendly, or friends-hangout?

nothing specific

Thank you for the information! What would be your budget for two people, in Rupees?

no budget constraint

Thank you for the details! Here's the final prompt:

```python
{'location': 'Banashankari', 'prompt': 'Looking for Indian restaurants in Banashankari with no dietary restrictions, no specific ambience preference, and no budget constraints. Top 5 distinct restaurants.'}
```

Thank you for providing all the information. Kindly wait, while I fetch the restaurant details: 

------------------ Top Restaurants as per your requirement -------------------
[
    {
        "Name": " Jalsa",
        "Add

##### Case2: For the Location which is present in database and specific user requirements

In [121]:
dialogue_mgmt_system()

"Hello! I'm here to help you find the perfect dining experience. To get started, could you please tell me the location where you're looking for restaurants?\n"

Banashankari

Thank you for sharing the location! Next, are you looking for any specific cuisine like Indian, Italian, Asian, Continental, Chinese, or Desserts?

south indian

Noted! Do you have any dietary restrictions, such as Veg, Non-Veg, Vegan, or Jain?

Pure Veg

Understood! For the ambience, would you prefer something family-friendly, romantic, kid-friendly, or friends-hangout?

family friendly

Thank you for the details! What budget do you have in mind for two people?

1000

Great! Are there any additional preferences or requirements you would like to include?

no

Thank you for all the information! Here's the final prompt:

```python
{'location': 'Banashankari', 'prompt': 'Looking for South Indian pure vegetarian restaurants in Banashankari, with a family-friendly ambience and a budget of 1000 Rupees for two people. Top 5 distinct restaurants.'}
```

Thank you for providing all the information. Kindly wait, while I fetch the restaurant details: 

------------------ Top Restaur

##### Case3: For the Location which is present in database and specific user requirements - Testing natural language processing capability

In [132]:
dialogue_mgmt_system()

"Hello! I'm here to help you find the perfect dining experience. To get started, could you please tell me the location where you're looking for restaurants?\n"

Banashankari

Thank you for sharing the location! Next, are you looking for any specific cuisine like Indian, Italian, Asian, Continental, Chinese, or Desserts?

Ice cream

Great choice! Ice cream falls under the Desserts category. Do you have any dietary restrictions, such as Veg, Non-Veg, Vegan, or Jain?

no

Noted! Now, for the ambience, would you prefer something family-friendly, romantic, kid-friendly, or friends-hangout?

nothing specific

Thank you for the clarification! Now, could you please provide the budget for two people in Rupees?

2000

Thank you for the details! Here’s the final prompt:

```python
{'location': 'Banashankari', 'prompt': 'Looking for top 5 distinct desserts restaurants, specifically ice cream, in Banashankari, with no dietary restrictions, no specific ambience preference, and a budget of 2000 Rupees for two people.'}
```

Thank you for providing all the information. Kindly wait, while I fetch the restaurant details: 

------------------ Top Restaurants as 

##### Case4: For the location which is not present in database - It should give the list of restaurants in the near by location

In [133]:
dialogue_mgmt_system()

"Hello! I'm here to help you find the perfect dining experience. To get started, could you please tell me the location where you're looking for restaurants?\n"

Gowdanapalya

Assistant: "Are you looking for any specific cuisine like Indian, Italian, Asian, Continental, Chinese or Desserts?"

Indian

Assistant: "Noted! Do you have any dietary restrictions, such as Veg, Non-Veg, Vegan, or Jain?"

no

Assistant: "Understood! For the ambience, would you prefer something family-friendly, romantic, kid-friendly, or friends-hangout?"

nothing specific

Assistant: "Thank you for the details! What would be your budget for two people?"

1000

Assistant: "Finally, are there any additional preferences or requirements that you would like to mention?"

no

Thank you for providing all the details! Here's the final prompt:

```python
{'location': 'Gowdanapalya', 'prompt': 'Looking for Indian restaurants in Gowdanapalya, with no dietary restrictions, no specific ambience preference, and a budget of 1000 for two people. Top 5 distinct restaurants.'}
```

Thank you for providing all the information. Kindly wait, while I fetch the restaurant details: 

No records

#### **Conclusion**:
RestaurantAssistAI demonstrates an end-to-end GenAI application for personalized restaurant recommendation.
</br>
The system combines structured restaurant data, unstructured review analysis, conversational requirement gathering, location-aware filtering, and LLM-based recommendation generation. Instead of relying on fixed rule-based filters, the chatbot allows users to express preferences naturally and maps those preferences to suitable restaurant options.
</br>
The application successfully demonstrates how OpenAI APIs and LangChain can be used to build a practical recommendation assistant that understands user intent, retrieves relevant candidates, and generates explainable recommendations.

#### **Future Enhancements**:

1. Scale feature extraction to the complete restaurant dataset.
2. Replace dataframe filtering with a vector database for semantic retrieval.
3. Add user feedback loops to improve future recommendations.
4. Integrate live restaurant availability and booking APIs.
5. Add personalization based on user history and previous preferences.
6. Deploy the chatbot as a web application using Streamlit or FastAPI.
7. Add guardrails for hallucination control and structured output validation.